# 01 · Bronze Initial Load

**Purpose:** Generate synthetic data and create Bronze Delta tables for the Customer360 lakehouse.

**Run:** ONCE manually — this bootstraps the entire Bronze layer.

**Compute:** Databricks serverless notebook.

**Output:** Creates 4 Delta tables in the `customer360` schema:
- `customer360.bronze_users`
- `customer360.bronze_transactions`
- `customer360.bronze_app_usage`
- `customer360.bronze_support_tickets`

## Step 1: Install Dependencies

Install the `faker` library to generate realistic synthetic data. The notebook kernel will restart after installation to load the new package.

In [0]:
%pip install faker --quiet
dbutils.library.restartPython()

## Step 2: Imports and Configuration

Import required libraries and set up configuration parameters:
- **Row counts:** Define the number of synthetic records for each table
- **Date range:** Set the time window for generated data (2023-2024)
- **Run ID:** Create a unique identifier to track which bootstrap run created each record

In [0]:
import uuid
import random
from datetime import datetime, timedelta

from faker import Faker
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField,
    StringType, IntegerType, DoubleType
)

fake   = Faker("en_IN")
spark  = SparkSession.builder.getOrCreate()

# Row counts — adjust freely, these are realistic starting volumes
N_USERS = 500
N_TXN   = 3_000
N_SES   = 2_500
N_TKT   = 800

DATE_START = datetime(2023, 1, 1)
DATE_END   = datetime(2024, 12, 31)

# Stamp every row with the run that created it — lineage tracing
RUN_ID = "BOOTSTRAP_" + datetime.utcnow().strftime("%Y%m%d_%H%M%S")

print(f"RUN_ID  : {RUN_ID}")
print(f"Rows    : users={N_USERS}  txn={N_TXN}  sessions={N_SES}  tickets={N_TKT}")

## Step 3: Define Helper Functions and Lookup Lists

Create reusable helper functions and lookup lists for generating realistic synthetic data:
- **Lookup lists:** Cities, customer segments, product categories, payment methods, etc.
- **Helper functions:** Random date generation, date formatting, unique ID generation

In [0]:
CITIES    = ["Mumbai","Delhi","Bangalore","Chennai","Hyderabad",
             "Pune","Kolkata","Ahmedabad","Jaipur","Coimbatore"]
SEGMENTS  = ["Premium","Standard","Basic","Trial"]
CATS      = ["Electronics","Fashion","Groceries","Travel",
             "Entertainment","Health","Sports"]
PAYMENTS  = ["Credit Card","Debit Card","UPI","Net Banking","Wallet"]
DEVICES   = ["Mobile","Desktop","Tablet"]
PLATFORMS = ["iOS","Android","Web"]
ISSUES    = ["Payment Failure","Delivery Issue","Login Problem",
             "Refund Request","Product Defect","Account Query"]
PRIORITIES = ["Low","Medium","High","Critical"]

def rand_date(start=DATE_START, end=DATE_END) -> datetime:
    delta = int((end - start).total_seconds())
    return start + timedelta(seconds=random.randint(0, delta))

def fmt(dt: datetime) -> str:
    return dt.strftime("%Y-%m-%d %H:%M:%S")

def short_id(prefix: str = "") -> str:
    return prefix + str(uuid.uuid4())[:8].upper()

print("Helper functions and lookup lists defined successfully")

## Step 4: Create Database Schema

Create the `customer360` schema (database) to hold all Bronze layer tables. This schema will serve as the foundation for the Customer 360 lakehouse architecture.

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS customer360
COMMENT 'Customer 360 lakehouse — Bronze / Silver / Gold layers';

In [0]:
# Confirm schema exists
spark.sql("SHOW SCHEMAS").filter("databaseName = 'customer360'").show()

## Step 5: Generate Bronze Users

Create the `bronze_users` table with synthetic customer data:
- **User attributes:** Name, age, gender, email, location
- **Business attributes:** Customer segment, signup date, active status
- **Lineage:** Pipeline run ID for tracking data provenance

This table serves as the foundation for customer profiles in the lakehouse.

In [0]:
random.seed(42)   # seed only for bootstrap — incremental runs are unseeded

user_ids = [short_id() for _ in range(N_USERS)]
rows = []

for uid in user_ids:
    signup = rand_date(DATE_START, datetime(2024, 6, 30))
    rows.append((
        uid,
        fake.name(),
        random.randint(18, 65),
        random.choice(["Male", "Female", "Other"]),
        signup.strftime("%Y-%m-%d"),
        random.choice(CITIES),
        "India",
        fake.email(),
        random.choice(SEGMENTS),
        random.choices([1, 0], weights=[75, 25])[0],  # 75% active
        RUN_ID,
    ))

users_schema = StructType([
    StructField("user_id",          StringType(),  False),
    StructField("name",             StringType(),  True),
    StructField("age",              IntegerType(), True),
    StructField("gender",           StringType(),  True),
    StructField("signup_date",      StringType(),  True),
    StructField("city",             StringType(),  True),
    StructField("country",          StringType(),  True),
    StructField("email",            StringType(),  True),
    StructField("segment",          StringType(),  True),
    StructField("is_active",        IntegerType(), True),
    StructField("pipeline_run_id",  StringType(),  True),
])

(
    spark.createDataFrame(rows, schema=users_schema)
    .write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("customer360.bronze_users")
)

count = spark.table("customer360.bronze_users").count()
print(f"customer360.bronze_users created — {count} rows")

## Step 6: Generate Bronze Transactions

Create the `bronze_transactions` table with synthetic transaction data:
- **Transaction details:** Amount, product, category, payment method
- **Business metrics:** Status, discount, transaction timestamp
- **Realistic distribution:** Most transactions are small (50-500), with some large ones creating natural skew

This table captures all customer purchase activity.

In [0]:
txn_rows = []

for _ in range(N_TXN):
    ts = rand_date()
    # Realistic skew: most txns are small, few are large
    amount = round(random.choices(
        population=[
            random.uniform(50,    500),
            random.uniform(500,  5_000),
            random.uniform(5_000, 50_000),
        ],
        weights=[60, 30, 10],
    )[0], 2)

    txn_rows.append((
        short_id("TXN"),
        random.choice(user_ids),
        amount,
        fmt(ts),
        short_id("PRD"),
        random.choice(CATS),
        random.choice(PAYMENTS),
        random.choice(CITIES),
        "India",
        random.choices(
            ["success", "failed", "refunded"],
            weights=[85, 10, 5]
        )[0],
        random.choice([0, 0, 0, 5, 10, 15, 20]),  # 0 most common
        random.choice(PLATFORMS),
        RUN_ID,
    ))

txn_schema = StructType([
    StructField("transaction_id",        StringType(),  False),
    StructField("user_id",               StringType(),  False),
    StructField("amount",                DoubleType(),  True),
    StructField("transaction_timestamp", StringType(),  True),
    StructField("product_id",            StringType(),  True),
    StructField("category",              StringType(),  True),
    StructField("payment_method",        StringType(),  True),
    StructField("city",                  StringType(),  True),
    StructField("country",               StringType(),  True),
    StructField("status",                StringType(),  True),
    StructField("discount_pct",          IntegerType(), True),
    StructField("platform",              StringType(),  True),
    StructField("pipeline_run_id",       StringType(),  True),
])

(
    spark.createDataFrame(txn_rows, schema=txn_schema)
    .write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("customer360.bronze_transactions")
)

count = spark.table("customer360.bronze_transactions").count()
print(f"customer360.bronze_transactions created — {count} rows")

## Step 7: Generate Bronze App Usage

Create the `bronze_app_usage` table with synthetic user session data:
- **Session metrics:** Duration, pages visited, actions taken
- **Device context:** Device type, platform (iOS/Android/Web)
- **Engagement signals:** Bounce flag (sessions under 1 minute)

This table tracks user engagement and app interaction patterns.

In [0]:
session_rows = []

for _ in range(N_SES):
    start   = rand_date()
    dur     = timedelta(minutes=random.randint(1, 90))
    end     = start + dur

    session_rows.append((
        short_id("SES"),
        random.choice(user_ids),
        fmt(start),
        fmt(end),
        round(dur.total_seconds() / 60, 1),
        random.randint(1, 25),
        random.randint(0, 15),
        random.choice(DEVICES),
        random.choice(PLATFORMS),
        1 if dur.total_seconds() < 60 else 0,   # bounce = under 1 min
        RUN_ID,
    ))

session_schema = StructType([
    StructField("session_id",            StringType(),  False),
    StructField("user_id",               StringType(),  False),
    StructField("session_start",         StringType(),  True),
    StructField("session_end",           StringType(),  True),
    StructField("session_duration_mins", DoubleType(),  True),
    StructField("pages_visited",         IntegerType(), True),
    StructField("actions_taken",         IntegerType(), True),
    StructField("device_type",           StringType(),  True),
    StructField("platform",              StringType(),  True),
    StructField("is_bounce",             IntegerType(), True),
    StructField("pipeline_run_id",       StringType(),  True),
])

(
    spark.createDataFrame(session_rows, schema=session_schema)
    .write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("customer360.bronze_app_usage")
)

count = spark.table("customer360.bronze_app_usage").count()
print(f"customer360.bronze_app_usage created — {count} rows")

## Step 8: Generate Bronze Support Tickets

Create the `bronze_support_tickets` table with synthetic support ticket data:
- **Ticket details:** Issue type, priority, status
- **Time metrics:** Created timestamp, resolved timestamp, resolution hours
- **Quality signals:** Satisfaction score (1-5)
- **Churn signal:** Intentionally creates 80 "heavy complainers" with disproportionate ticket volume

This table is crucial for customer satisfaction analysis and churn prediction.

In [0]:
# Bias: 80 "heavy complainers" appear disproportionately
# This creates realistic churn signal data in downstream Gold layer
heavy_complainers = random.sample(user_ids, 80)
ticket_pool       = heavy_complainers * 5 + user_ids

ticket_rows = []

for _ in range(N_TKT):
    created  = rand_date()
    resolved = (
        created + timedelta(hours=random.randint(1, 72))
        if random.random() > 0.2   # 80% get resolved
        else None
    )

    ticket_rows.append((
        short_id("TKT"),
        random.choice(ticket_pool),
        random.choice(ISSUES),
        random.choice(PRIORITIES),
        fmt(created),
        fmt(resolved) if resolved else None,
        round((resolved - created).total_seconds() / 3600, 1) if resolved else None,
        "closed" if resolved else "open",
        random.randint(1, 5) if resolved else None,
        RUN_ID,
    ))

ticket_schema = StructType([
    StructField("ticket_id",          StringType(),  False),
    StructField("user_id",            StringType(),  False),
    StructField("issue_type",         StringType(),  True),
    StructField("priority",           StringType(),  True),
    StructField("created_at",         StringType(),  True),
    StructField("resolved_at",        StringType(),  True),
    StructField("resolution_hours",   DoubleType(),  True),
    StructField("status",             StringType(),  True),
    StructField("satisfaction_score", IntegerType(), True),
    StructField("pipeline_run_id",    StringType(),  True),
])

(
    spark.createDataFrame(ticket_rows, schema=ticket_schema)
    .write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("customer360.bronze_support_tickets")
)

count = spark.table("customer360.bronze_support_tickets").count()
print(f"customer360.bronze_support_tickets created — {count} rows")

## Step 9: Verify All Tables

Confirm that all 4 Bronze tables were created successfully:
- Show all tables in the `customer360` schema
- Display row counts and column counts for each table
- Verify the pipeline run ID for data lineage tracing

In [0]:
%sql
SHOW TABLES IN customer360;

In [0]:
tables = [
    "customer360.bronze_users",
    "customer360.bronze_transactions",
    "customer360.bronze_app_usage",
    "customer360.bronze_support_tickets",
]

print("=" * 55)
print(f"  Bootstrap complete | RUN_ID: {RUN_ID}")
print("=" * 55)
for t in tables:
    n = spark.table(t).count()
    cols = len(spark.table(t).columns)
    print(f"  {t:<45} {n:>5} rows  {cols} cols")
print("=" * 55)
print("  Next: run 02_incremental_generator on a schedule")

## Step 10: Quick Data Preview

Preview sample data from each table to verify data quality and realistic distributions.

In [0]:
%sql
-- Preview sample users with all attributes
SELECT * 
FROM customer360.bronze_users 
LIMIT 5;

In [0]:
%sql
-- Preview transaction status distribution and amount statistics
SELECT
    status,
    COUNT(*)              AS txn_count,
    ROUND(AVG(amount), 2) AS avg_amount,
    ROUND(MIN(amount), 2) AS min_amount,
    ROUND(MAX(amount), 2) AS max_amount
FROM customer360.bronze_transactions
GROUP BY status
ORDER BY txn_count DESC;

In [0]:
%sql
-- Confirm pipeline_run_id lineage across all 4 tables
SELECT 'bronze_users' AS tbl, pipeline_run_id, COUNT(*) AS rows 
FROM customer360.bronze_users 
GROUP BY pipeline_run_id

UNION ALL

SELECT 'bronze_transactions' AS tbl, pipeline_run_id, COUNT(*) AS rows 
FROM customer360.bronze_transactions 
GROUP BY pipeline_run_id

UNION ALL

SELECT 'bronze_app_usage' AS tbl, pipeline_run_id, COUNT(*) AS rows 
FROM customer360.bronze_app_usage 
GROUP BY pipeline_run_id

UNION ALL

SELECT 'bronze_support_tickets' AS tbl, pipeline_run_id, COUNT(*) AS rows 
FROM customer360.bronze_support_tickets 
GROUP BY pipeline_run_id

ORDER BY tbl;